# 💬 UltraGPT Interactive Chatbot
Welcome to the interactive inference and chatbot playground for **UltraGPT**!

This notebook provides a premium, dark-mode chat interface built directly inside your Jupyter environment using `ipywidgets`. You can chat with your trained model in real-time, adjust generation hyperparameters (like temperature, top-k, and top-p), and modify the system instructions on the fly.

### ⚡ Powered by:
- **Autoregressive KV-Cache Sampler** (`UltraGPTSampler`): For O(1) step latency.
- **XLA compilation** (`jit_compile=True`): For hardware-accelerated token generation.
- **Dynamic Streaming**: Watch tokens print live as they are predicted.

In [ ]:
import sys
import os
import tensorflow as tf
import numpy as np

# Ensure the project root is in the system path so we can import local modules
project_root = os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import UltraGPTConfig
from models.transformer import UltraGPT
from data_pipeline.pipeline import TiktokenWrapper
from inference.sampler import UltraGPTSampler
from inference.chatbot import UltraGPTChatbot

print(f"✅ TensorFlow Version: {tf.__version__}")
print(f"✅ GPU Available: {tf.config.list_physical_devices('GPU')}")
print(f"✅ NumPy Version: {np.__version__}")

In [ ]:
# Initialize configuration matching the trained architecture in notebook.ipynb
config = UltraGPTConfig(
    batch_size=4,
    d_model=1000,
    n_heads=100,
    n_kv_heads=20,
    n_layers=6,
    block_size=128,
    dropout_rate=0.05,
    learning_rate=3e-4,
    warmup_steps=200,
    max_steps=100,
    weight_decay=0.01,
    temperature=0.10,
    top_k=0,
    top_p=0.10,
    max_gen_length=128,
)

print("🏗️ Initializing model structure...")
model = UltraGPT(config, name="ultra_gpt")

# Perform dummy forward pass to build Keras variables
dummy_input = tf.zeros((1, config.block_size), dtype=tf.int32)
_ = model(dummy_input, training=False)

# Find and load the latest weights file
weight_paths = [
    "output/notebook_weights.weights.h5",
    "output/notebook_checkpoints/ultra_gpt_notebook_latest.weights.h5",
    "output/final_weights.weights.h5",
    "output/checkpoints/ultra_gpt_latest.weights.h5"
]

loaded = False
for path in weight_paths:
    if os.path.exists(path):
        print(f"💾 Loading weights from {path}...")
        model.load_weights(path)
        print("✅ Model weights loaded successfully!")
        loaded = True
        break

if not loaded:
    print("⚠️ Warning: No weights file found. Model will produce un-trained predictions.")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# ── 1. Sampler and Chatbot Initialization ─────────────────────────────────
tokenizer = TiktokenWrapper()
sampler = UltraGPTSampler(model, tokenizer, config)
chatbot = UltraGPTChatbot(sampler, system_prompt="You are a helpful, concise AI assistant named UltraGPT.")

# ── 2. Premium CSS Styling ─────────────────────────────────────────────────
style_css = """
<style>
.chat-window {
    background-color: #121214;
    border-radius: 12px;
    border: 1px solid #2b2b36;
    padding: 20px;
    max-height: 480px;
    overflow-y: auto;
    font-family: 'Inter', -apple-system, sans-serif;
}
.chat-msg {
    margin-bottom: 16px;
    display: flex;
    flex-direction: column;
}
.chat-msg.user {
    align-items: flex-end;
}
.chat-msg.assistant {
    align-items: flex-start;
}
.msg-label {
    font-size: 11px;
    color: #808095;
    margin-bottom: 4px;
    font-weight: 600;
}
.msg-bubble {
    padding: 10px 16px;
    border-radius: 14px;
    max-width: 80%;
    font-size: 14px;
    line-height: 1.45;
    word-break: break-word;
    box-shadow: 0 4px 10px rgba(0, 0, 0, 0.15);
}
.msg-bubble.user {
    background: linear-gradient(135deg, #6366f1, #8b5cf6);
    color: #ffffff;
    border-bottom-right-radius: 2px;
}
.msg-bubble.assistant {
    background-color: #20202a;
    color: #e2e8f0;
    border-bottom-left-radius: 2px;
}
.system-msg {
    text-align: center;
    color: #f59e0b;
    font-size: 12px;
    margin: 8px 0;
    font-style: italic;
}
</style>
"""

# ── 3. Widget Elements Setup ──────────────────────────────────────────────
chat_history_html = []

history_output = widgets.HTML(
    value=style_css + "<div class='chat-window'><div class='system-msg'>Start typing below to chat with UltraGPT...</div></div>"
)

input_box = widgets.Text(
    value="",
    placeholder="Type a message and press Enter...",
    layout=widgets.Layout(width='80%')
)

send_button = widgets.Button(
    description="Send",
    button_style="primary",
    icon="paper-plane",
    layout=widgets.Layout(width='18%')
)

clear_button = widgets.Button(
    description="Clear History",
    button_style="danger",
    icon="trash",
    layout=widgets.Layout(width='100%', margin='10px 0 0 0')
)

# Generation Hyperparameters Panel
temp_slider = widgets.FloatSlider(value=0.10, min=0.1, max=1.5, step=0.05, description="Temperature")
top_k_slider = widgets.IntSlider(value=0, min=0, max=100, step=1, description="Top-K")
top_p_slider = widgets.FloatSlider(value=0.10, min=0.1, max=1.0, step=0.05, description="Top-P")
max_tokens_slider = widgets.IntSlider(value=128, min=16, max=256, step=8, description="Max Tokens")
system_prompt_input = widgets.Text(
    value=chatbot.system_prompt or "", 
    placeholder="System instructions...",
    description="Sys Prompt",
    layout=widgets.Layout(width='100%')
)

# ── 4. Control Logic & Stream Update ──────────────────────────────────────
def update_chat_display(live_assistant_text=None):
    content = "".join(chat_history_html)
    if live_assistant_text is not None:
        content += f"""
        <div class='chat-msg assistant'>
            <div class='msg-label'>Assistant</div>
            <div class='msg-bubble assistant'>{live_assistant_text}</div>
        </div>
        """
    history_output.value = style_css + f"<div class='chat-window'>{content}</div>"

def on_send(b=None):
    user_text = input_box.value.strip()
    if not user_text:
        return
    
    # Disable inputs while generating
    input_box.disabled = True
    send_button.disabled = True
    
    chatbot.system_prompt = system_prompt_input.value.strip() or None
    
    # Append User Message
    user_html = f"""
    <div class='chat-msg user'>
        <div class='msg-label'>User</div>
        <div class='msg-bubble user'>{user_text}</div>
    </div>
    """
    chat_history_html.append(user_html)
    update_chat_display(live_assistant_text="...")
    input_box.value = ""
    
    try:
        # Stream Response
        response_generator = chatbot.respond(
            user_message=user_text,
            max_new_tokens=max_tokens_slider.value,
            temperature=temp_slider.value,
            top_k=top_k_slider.value,
            top_p=top_p_slider.value,
            stream=True
        )
        
        full_reply = ""
        for chunk in response_generator:
            full_reply += chunk
            update_chat_display(live_assistant_text=full_reply)
        
        # Append final Assistant Message
        assistant_html = f"""
        <div class='chat-msg assistant'>
            <div class='msg-label'>Assistant</div>
            <div class='msg-bubble assistant'>{full_reply}</div>
        </div>
        """
        chat_history_html.append(assistant_html)
        
    except Exception as e:
        error_html = f"<div class='system-msg' style='color: #ef4444;'>Error: {str(e)}</div>"
        chat_history_html.append(error_html)
        
    update_chat_display()
    
    # Re-enable inputs
    input_box.disabled = False
    send_button.disabled = False

def on_clear(b):
    chatbot.clear_history()
    chat_history_html.clear()
    chat_history_html.append("<div class='system-msg'>Conversation history cleared.</div>")
    update_chat_display()

send_button.on_click(on_send)
input_box.on_submit(on_send)
clear_button.on_click(on_clear)

# ── 5. Render Panel ───────────────────────────────────────────────────────
sidebar = widgets.VBox([
    widgets.HTML("<h4>⚙️ Settings</h4>"),
    temp_slider,
    top_k_slider,
    top_p_slider,
    max_tokens_slider,
    system_prompt_input,
    clear_button
], layout=widgets.Layout(width='32%', padding='12px', border='1px solid #2b2b36', border_radius='8px', margin='0 0 0 10px'))

input_area = widgets.HBox([input_box, send_button], layout=widgets.Layout(margin='10px 0 0 0'))
main_chat = widgets.VBox([history_output, input_area], layout=widgets.Layout(width='66%'))

chat_ui = widgets.HBox([main_chat, sidebar])

display(HTML("<h2>💬 Interactive Chat Playground</h2>"))
display(chat_ui)